In [0]:
%sql
-- =============================================================================
-- Patient HCP Visit Summary View - Top 5 based on 3-Year activity
-- =============================================================================
-- Purpose: Identify key healthcare providers for MPSII patients (Elaprase GTM)
-- Windows:
--   5 yrs: 2020-08-01..2025-07-31
--   3 yrs: 2022-08-01..2025-07-31
--   2 yrs: 2023-08-01..2025-07-31
-- Ranking basis:
--   Top 5 by 3-year visit count (Dx + Tx), tie-break: last visit (DESC), NPI (ASC)
-- Outputs include 3yr ranking, but visit counts and last visit dates use 5yr window
-- New additions: Most recent Tx HCP in 2yr, Latest treatment date in 5yr
-- =============================================================================
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ----------------------------------------------------------
-- Claims universes (5y, 3y, 2y) Dx + Tx, with NPIs
-- ----------------------------------------------------------
all_dx_claims_5yr AS (
    SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE IN ('E761','E763')
     AND TRANSACTION_STATUS = 'PAID'
     AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
     AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),
all_tx_claims_5yr AS (
    SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND 
      PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI,REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                           '38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),
all_claims_5yr AS (
    SELECT *, 'DX Claim' FROM all_dx_claims_5yr
    UNION
    SELECT *, 'Tx Claim' FROM all_tx_claims_5yr
)
SELECT * FROM all_claims_5yr
WHERE patient_id = 'XBV2CVLN'


-- all_claims_3yr AS (
--     SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--       AND SERVICE_DATE BETWEEN '2022-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     UNION
--     SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE IN ('E761','E763')
--       AND TRANSACTION_STATUS = 'PAID'
--       AND FILL_DATE BETWEEN '2022-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     UNION
--     SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events 
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND SERVICE_DATE BETWEEN '2022-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     UNION
--     SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2022-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     UNION
--     SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events 
--     WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                              '38206','38230','38232','38240','38241','38242','38243','38250')
--       AND SERVICE_DATE BETWEEN '2022-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- )
-- SELECT a.*,p1.first_name, p1.last_name, p1.PRIMARY_SPECIALTY from all_claims_5yr a
-- INNER JOIN com_edp_prd.com_raw.kom_providers p1
-- ON p1.NPI = a.NPI
-- where patient_id = 'B53617TZ'


-- -- -- ----------------------------------------------------------
-- -- -- First Dx / First Tx (5y)
-- -- -- ----------------------------------------------------------
-- -- first_dx_hcp_ranked AS (
-- --     SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
-- --            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
-- --     FROM all_dx_claims_5yr
-- --     WHERE NPI IS NOT NULL
-- --     GROUP BY PATIENT_ID, NPI
-- -- )
-- -- SELECT * from first_dx_hcp_ranked
-- -- where patient_id = '1TKZ01BN'

-- -- first_dx_hcp AS (
-- --     SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
-- --     FROM first_dx_hcp_ranked
-- --     WHERE rn = 1
-- -- ),
-- -- first_dx_hcp_5yr_stats AS (
-- --     SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
-- --            COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
-- --            MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
-- --     FROM first_dx_hcp fdh
-- --     LEFT JOIN all_claims_5yr ac
-- --       ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
-- --     GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
-- -- ),

-- -- SELECT * from first_dx_hcp_5yr_stats
-- -- where patient_id = '6XZ63TZJ'

-- -- first_tx_hcp_ranked AS (
-- --     SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
-- --            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
-- --     FROM all_tx_claims_5yr
-- --     WHERE NPI IS NOT NULL
-- --     GROUP BY PATIENT_ID, NPI
-- -- ),
-- -- first_tx_hcp AS (
-- --     SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
-- --     FROM first_tx_hcp_ranked
-- --     WHERE rn = 1
-- -- ),
-- -- first_tx_hcp_5yr_stats AS (
-- --     SELECT fth.PATIENT_ID, fth.first_tx_hcp,
--            COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
--            MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
--     FROM first_tx_hcp fth
--     LEFT JOIN all_claims_5yr ac
--       ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
--     GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
-- ),
-- first_tx_hcp_5yr_tx_only AS (
--     SELECT fth.PATIENT_ID, fth.first_tx_hcp,
--            COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
--     FROM first_tx_hcp fth
--     LEFT JOIN all_tx_claims_5yr tx
--       ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
--     GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
-- ),

-- -- ----------------------------------------------------------
-- -- Most-seen ranking (Top 5 by 3y) with 5y counts + 5y last-visit
-- -- ----------------------------------------------------------
-- most_seen_3yr_ranking AS (
--     SELECT PATIENT_ID,
--            NPI,
--            COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
--            MAX(FILL_DATE) AS last_visit_3yr,
--            ROW_NUMBER() OVER (
--              PARTITION BY PATIENT_ID
--              ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
--                       MAX(FILL_DATE) DESC,
--                       NPI ASC
--            ) AS rank
--     FROM all_claims_3yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),
-- most_seen_combined_stats AS (
--     SELECT
--         ms3.PATIENT_ID,
--         ms3.NPI,
--         ms3.rank,
--         ms3.visit_count_3yr,
--         ms3.last_visit_3yr,
--         COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
--         MAX(ac5.FILL_DATE)          AS last_visit_5yr
--     FROM most_seen_3yr_ranking ms3
--     LEFT JOIN all_claims_5yr ac5
--       ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
--     WHERE ms3.rank <= 5
--     GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
-- ),

-- -- ----------------------------------------------------------
-- -- Historical First Dates (No Date Restrictions)
-- -- ----------------------------------------------------------
-- historical_first_dx AS (
--     SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
--     FROM (
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE DIAGNOSIS_CODES LIKE '%E761%'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE DIAGNOSIS_CODE = 'E761'
--           AND TRANSACTION_STATUS = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE DIAGNOSIS_CODES LIKE '%E763%'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE DIAGNOSIS_CODE = 'E763'
--           AND TRANSACTION_STATUS = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     ) all_dx
--     GROUP BY PATIENT_ID
-- ),
-- historical_first_tx AS (
--     SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
--     FROM (
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                  '38206','38230','38232','38240','38241','38242','38243','38250')
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     ) all_tx
--     GROUP BY PATIENT_ID
-- ),
-- latest_claim AS (
--     SELECT PATIENT_ID, MAX(FILL_DATE) AS latest_claim_date
--     FROM all_claims_5yr
--     GROUP BY PATIENT_ID
-- ),
-- -- ----------------------------------------------------------
-- -- Most Recent Treatment HCP (2-year window)
-- -- ----------------------------------------------------------
-- all_tx_claims_2yr AS (
--     SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events 
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     UNION
--     SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     UNION
--     SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events 
--     WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                              '38206','38230','38232','38240','38241','38242','38243','38250')
--       AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- ),
-- SELECT a.*,p1.first_name, p1.last_name, p1.PRIMARY_SPECIALTY from all_tx_claims_2yr a
-- INNER JOIN com_edp_prd.com_raw.kom_providers p1
-- ON p1.NPI = a.NPI
-- where patient_id = '6XZ63TZJ'

-- most_recent_tx_hcp_2yr_ranked AS (
--     SELECT PATIENT_ID, NPI, MAX(FILL_DATE) AS most_recent_tx_date,
--            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MAX(FILL_DATE) DESC, NPI ASC) AS rn
--     FROM all_tx_claims_2yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),
-- -- SELECT a.*,p1.first_name, p1.last_name, p1.PRIMARY_SPECIALTY from most_recent_tx_hcp_2yr_ranked a
-- INNER JOIN com_edp_prd.com_raw.kom_providers p1
-- ON p1.NPI = a.NPI
-- where patient_id = '6XZ63TZJ'
-- most_recent_tx_hcp_2yr AS (
--     SELECT PATIENT_ID, 
--            NPI AS most_recent_tx_hcp_2yr,
--            most_recent_tx_date AS most_recent_tx_hcp_2yr_date
--     FROM most_recent_tx_hcp_2yr_ranked
--     WHERE rn = 1
-- ),
-- -- ----------------------------------------------------------
-- -- Latest Treatment Date (5-year window)
-- -- ----------------------------------------------------------
-- latest_treatment_date_5yr AS (
--     SELECT PATIENT_ID, MAX(FILL_DATE) AS latest_treatment_date_5yr
--     FROM all_tx_claims_5yr
--     GROUP BY PATIENT_ID
-- ),
-- -- Patient Level Information
-- patient_demographics AS (
--     SELECT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER
--     FROM com_edp_prd.com_raw.kom_patient_demographics
-- ),
-- patient_geography AS (
--     SELECT PATIENT_ID, patient_state
--     FROM com_edp_prd.com_raw.kom_patient_geography
--     WHERE VALID_TO_DATE > CURRENT_DATE()
-- )

-- -- -----------------------------
-- -- Final output
-- -- -----------------------------
-- SELECT
--     ep.PATIENT_ID,
--     pd.PATIENT_YOB,
--     YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
--     pd.PATIENT_GENDER,
--     pg.patient_state,   
--     -- Historical First Dates
--     hfdx.incidence_date,
--     hftx.first_incidence_treatment_date,
--     lc.latest_claim_date,
    
--     -- Most Recent Treatment Info (2-year window)
--     mrtx2.most_recent_tx_hcp_2yr,
--     mrtx2.most_recent_tx_hcp_2yr_date,
    
--     -- Latest Treatment Date (5-year window)
--     ltx5.latest_treatment_date_5yr,
    
--     -- First Dx HCP
--     fdh.first_dx_hcp AS first_dx_hcp_5yr,
--     COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
--     fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

--     -- First Tx HCP
--     fth.first_tx_hcp AS first_tx_hcp_5yr,
--     COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
--     COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
--     fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

--     -- Most Seen HCP #1..#5 (ranked by 3-year activity, showing 5-year visit metrics)
    
--     -- #1 Most Seen
--     ms1.NPI  AS most_seen_hcp1_3yr_ranked,
--     COALESCE(ms1.visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
--     ms1.last_visit_5yr AS most_seen_hcp1_last_visit_5yr,
    
--     -- #2 Most Seen
--     ms2.NPI  AS most_seen_hcp2_3yr_ranked,
--     COALESCE(ms2.visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
--     ms2.last_visit_5yr AS most_seen_hcp2_last_visit_5yr,

--     -- #3 Most Seen
--     ms3.NPI  AS most_seen_hcp3_3yr_ranked,
--     COALESCE(ms3.visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
--     ms3.last_visit_5yr AS most_seen_hcp3_last_visit_5yr,
    
--     -- #4 Most Seen
--     ms4.NPI  AS most_seen_hcp4_3yr_ranked,
--     COALESCE(ms4.visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
--     ms4.last_visit_5yr AS most_seen_hcp4_last_visit_5yr,
    
--     -- #5 Most Seen
--     ms5.NPI  AS most_seen_hcp5_3yr_ranked,
--     COALESCE(ms5.visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
--     ms5.last_visit_5yr AS most_seen_hcp5_last_visit_5yr

-- FROM eligible_patients ep
-- LEFT JOIN patient_demographics pd          ON ep.PATIENT_ID = pd.PATIENT_ID
-- LEFT JOIN patient_geography pg             ON ep.PATIENT_ID = pg.PATIENT_ID
-- LEFT JOIN historical_first_dx hfdx         ON ep.PATIENT_ID = hfdx.PATIENT_ID
-- LEFT JOIN historical_first_tx hftx         ON ep.PATIENT_ID = hftx.PATIENT_ID
-- LEFT JOIN latest_claim lc                  ON ep.PATIENT_ID = lc.PATIENT_ID
-- LEFT JOIN most_recent_tx_hcp_2yr mrtx2     ON ep.PATIENT_ID = mrtx2.PATIENT_ID
-- LEFT JOIN latest_treatment_date_5yr ltx5   ON ep.PATIENT_ID = ltx5.PATIENT_ID
-- LEFT JOIN first_dx_hcp fdh                 ON ep.PATIENT_ID = fdh.PATIENT_ID
-- LEFT JOIN first_dx_hcp_5yr_stats fdhs      ON ep.PATIENT_ID = fdhs.PATIENT_ID
-- LEFT JOIN first_tx_hcp fth                 ON ep.PATIENT_ID = fth.PATIENT_ID
-- LEFT JOIN first_tx_hcp_5yr_stats fths      ON ep.PATIENT_ID = fths.PATIENT_ID
-- LEFT JOIN first_tx_hcp_5yr_tx_only fthtx   ON ep.PATIENT_ID = fthtx.PATIENT_ID
-- LEFT JOIN most_seen_combined_stats ms1     ON ep.PATIENT_ID = ms1.PATIENT_ID AND ms1.rank = 1
-- LEFT JOIN most_seen_combined_stats ms2     ON ep.PATIENT_ID = ms2.PATIENT_ID AND ms2.rank = 2
-- LEFT JOIN most_seen_combined_stats ms3     ON ep.PATIENT_ID = ms3.PATIENT_ID AND ms3.rank = 3
-- LEFT JOIN most_seen_combined_stats ms4     ON ep.PATIENT_ID = ms4.PATIENT_ID AND ms4.rank = 4
-- LEFT JOIN most_seen_combined_stats ms5     ON ep.PATIENT_ID = ms5.PATIENT_ID AND ms5.rank = 5
-- ORDER BY ep.PATIENT_ID;

-- -- Create the table from the temp view
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360 AS
-- SELECT * FROM patient_hcp_visit_summary;


In [0]:
%sql
SELECT patient_id, PRIMARY_HCP_NPI FROM
com_edp_prd.cmpa_insights_internal_schema.primary_hcp

In [0]:
%sql
-- Using all codes (2 years)
create or replace temporary view mpsii_treatment_table as
SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
          --  WHERE PROCEDURE_CODE IN ('J1743')
)
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31';

In [0]:
%sql
create or replace temporary view mpsii_diagnosis_table as 
with mpsii_1dx_specified as (
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
mpsii_2dx_specified as (
  select *
  from mpsii_1dx_specified
  where patient_id in (select a.patient_id from mpsii_1dx_specified as a group by a.patient_id having count(distinct a.fill_date) >= 2)
),
mpsii_1dx_unspecified as (
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
mpsii_2dx_unspecified as (
  select *
  from mpsii_1dx_unspecified
  where patient_id in (select distinct a.patient_id from mpsii_1dx_unspecified as a group by a.patient_id having count(distinct a.fill_date) >= 2) 
),
mpsii_2dx_specified_tx as (
  select *
  from mpsii_treatment_table where patient_id in (select distinct a.patient_id from mpsii_2dx_specified as a)
),
incremental_patient as (
  select distinct patient_id
  from mpsii_2dx_unspecified
  where patient_id in (select distinct a.patient_id from mpsii_treatment_table as a where a.code in ('54092070001','540920700','J1743'))
  and patient_id not in (select distinct b.patient_id from mpsii_2dx_specified_tx as b)
),
all_dx_patients_claims as (
  select * from mpsii_2dx_specified
  union
  select * from mpsii_1dx_unspecified where patient_id in (select distinct a.patient_id from incremental_patient as a) 
)
select *
from all_dx_patients_claims

In [0]:
%sql
-- =============================================================================
-- MPSII Diagnosis Patients - Primary HCP Mapping (5 Year Lookback)
-- =============================================================================
-- Diagnosis Lookback: 2020-08-01 to 2025-07-31 (5 years)
-- 
-- Output: Patient ID + Primary HCP demographics, affiliations, and geography
-- =============================================================================
WITH pulling_specialities AS (
    SELECT 
        a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
            WHEN b.primary_specialty LIKE '%Genetic%' OR b.secondary_specialty LIKE '%Genetic%' 
                THEN 'Geneticist'
            WHEN b.primary_specialty LIKE '%Pediatrics%' 
                THEN 'Pediatrician'
            WHEN b.primary_specialty LIKE '%Psychiatry & Neurology%' OR 
                 b.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                 b.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 'Psychiatry & Neurology'
            WHEN b.primary_specialty LIKE '%Nurse Practitioner%' OR 
                 b.primary_specialty LIKE '%Physician Assistant%' 
                THEN 'NPPA'
            WHEN b.primary_specialty LIKE '%Internal Medicine%' OR 
                 b.secondary_specialty LIKE '%Internal Medicine%' 
                THEN 'PCP'
            WHEN b.primary_specialty LIKE '%Family Medicine%' OR 
                 b.secondary_specialty LIKE '%Family Medicine%' 
                THEN 'PCP'
            WHEN a.npi IS NULL 
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY 
    FROM (
        SELECT * FROM mpsii_diagnosis_table
    ) a 
    LEFT JOIN com_edp_prd.com_raw.kom_providers b ON a.npi = b.npi
),
primary_hcp AS (
    SELECT DISTINCT 
        n_pats AS patient_id, 
        npi 
    FROM (
        SELECT DISTINCT 
            NPI, 
            SPECIALTY,
            PATIENT_ID AS N_PATS, 
            Fill_date,
            KH_PLAN,
            HCO_PRIMARY_NPI,
            PLACE_OF_SERVICE,
            'DX' AS PATIENT_TYPE
        FROM (
            SELECT DISTINCT 
                PATIENT_ID,
                Fill_Date,
                KH_PLAN,
                HCO_PRIMARY_NPI,
                PLACE_OF_SERVICE,
                NPI, 
                SPECIALTY, 
                FINAL_HCP_RANK
            FROM (
                SELECT *, 
                       DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
                FROM (
                    SELECT *, 
                           MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
                    FROM (
                        SELECT *, 
                               RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC, NPI) AS HCP_RANK
                        FROM (
                            SELECT 
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE,
                                PRIORITY,
                                NO_OF_VISITS
                            FROM (
                                SELECT 
                                    PATIENT_ID,
                                    NPI,
                                    SPECIALTY,
                                    FILL_DATE,
                                    KH_PLAN,
                                    HCO_PRIMARY_NPI,
                                    PLACE_OF_SERVICE,
                                    CASE 
                                        WHEN SPECIALTY = 'Geneticist' THEN 1 
                                        WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                        WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                        WHEN SPECIALTY = 'PCP' THEN 4 
                                        WHEN SPECIALTY = 'NPPA' THEN 5
                                        WHEN SPECIALTY = 'Others' THEN 6 
                                        ELSE 7
                                    END AS PRIORITY,
                                    COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                                FROM pulling_specialities
                                GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE, KH_PLAN, HCO_PRIMARY_NPI, PLACE_OF_SERVICE
                            )
                        )
                    )
                ) 
            ) 
            -- WHERE HCP_RANK = 1
        )
    )
)

SELECT * from primary_hcp a
-- INNER JOIN com_edp_prd.com_raw.kom_providers p1
-- ON p1.NPI = a.NPI
where patient_id = '6XZ63TZJ'


-- pulling_demographics AS (
--     SELECT 
--         a.patient_id, 
--         a.npi AS primary_hcp, 
--         CONCAT(b.FIRST_NAME || ' ' || b.LAST_NAME) AS hcp_name, 
--         b.PRIMARY_SPECIALTY AS hcp_specialty, 
--         c.hco_npi, 
--         c.hco_name, 
--         c.final_zip, 
--         c.territory, 
--         c.region
--     FROM primary_hcp AS a
--     LEFT JOIN com_edp_prd.com_raw.kom_providers AS b 
--         ON a.npi = b.NPI AND b.PROVIDER_TYPE = 'INDIVIDUAL'
--     LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table AS c 
--         ON a.npi = c.hcp_npi
--     WHERE a.npi IS NOT NULL
-- ),
-- visit_counts AS (
--     SELECT 
--         patient_id, 
--         npi, 
--         COUNT(DISTINCT fill_date) AS visit_counts
--     FROM mpsii_diagnosis_table
--     WHERE npi IS NOT NULL
--     GROUP BY patient_id, npi
-- ),
-- final_joined AS (
--     SELECT 
--         a.*, 
--         b.visit_counts
--     FROM pulling_demographics AS a
--     LEFT JOIN visit_counts AS b 
--         ON a.primary_hcp = b.npi AND a.patient_id = b.patient_id
-- )
-- SELECT DISTINCT 
--     patient_id, 
--     primary_hcp AS primary_hcp_2yr, 
--     hcp_name AS primary_hcp_name_2yr, 
--     visit_counts AS primary_hcp_no_of_visits_2yr, 
--     hcp_specialty AS primary_hcp_specialty_2yr, 
--     hco_npi AS primary_hcp_hco_npi_2yr, 
--     hco_name AS primary_hcp_hco_name_2yr, 
--     final_zip AS primary_hcp_final_zip_2yr, 
--     territory AS primary_hcp_territory_2yr, 
--     region AS primary_hcp_region_2yr
-- FROM final_joined;